## Cell 1 — Install Ultralytics (YOLOv11)

In [1]:

# !pip install -q ultralytics


## Cell 2 — Paths and settings (full Kaggle DUT Anti-UAV dataset)

In [2]:

import os

KAGGLE_ROOT = r"C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1"

TRAIN_IMG_DIR = os.path.join(KAGGLE_ROOT, "train", "train", "img")
TRAIN_XML_DIR = os.path.join(KAGGLE_ROOT, "train", "train", "xml")
VAL_IMG_DIR   = os.path.join(KAGGLE_ROOT, "val", "val", "img")
VAL_XML_DIR   = os.path.join(KAGGLE_ROOT, "val", "val", "xml")
TEST_IMG_DIR  = os.path.join(KAGGLE_ROOT, "test", "test", "img")
TEST_XML_DIR  = os.path.join(KAGGLE_ROOT, "test", "test", "xml")

BASE_DIR = r"C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone"
YOLO_DATASET_DIR = os.path.join(BASE_DIR, "yolo_full_dataset")

YOLO_IMAGES_TRAIN = os.path.join(YOLO_DATASET_DIR, "images", "train")
YOLO_IMAGES_VAL   = os.path.join(YOLO_DATASET_DIR, "images", "val")
YOLO_LABELS_TRAIN = os.path.join(YOLO_DATASET_DIR, "labels", "train")
YOLO_LABELS_VAL   = os.path.join(YOLO_DATASET_DIR, "labels", "val")

for d in [YOLO_IMAGES_TRAIN, YOLO_IMAGES_VAL, YOLO_LABELS_TRAIN, YOLO_LABELS_VAL]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = ["UAV"]

for name, path in [("TRAIN_IMG_DIR", TRAIN_IMG_DIR), ("TRAIN_XML_DIR", TRAIN_XML_DIR),
                    ("VAL_IMG_DIR", VAL_IMG_DIR), ("VAL_XML_DIR", VAL_XML_DIR),
                    ("TEST_IMG_DIR", TEST_IMG_DIR), ("TEST_XML_DIR", TEST_XML_DIR)]:
    print(f"{name}: {path}  | exists: {os.path.exists(path)}")

print("\nClasses:", CLASS_NAMES)


TRAIN_IMG_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\train\train\img  | exists: True
TRAIN_XML_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\train\train\xml  | exists: True
VAL_IMG_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\val\val\img  | exists: True
VAL_XML_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\val\val\xml  | exists: True
TEST_IMG_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\test\test\img  | exists: True
TEST_XML_DIR: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\test\test\xml  | exists: True

Classes: ['UAV']


## Cell 3 — VOC XML -> YOLO format converter

In [3]:

from lxml import etree

def voc_to_yolo_lines(xml_path, class_names):
    # Returns a list of YOLO-format label lines (class x_center y_center w h, normalized)
    tree = etree.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    img_w = float(size.find("width").text)
    img_h = float(size.find("height").text)

    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_names:
            print(f"WARNING: unknown class '{name}' in {xml_path}, skipping this object")
            continue
        class_id = class_names.index(name)

        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    return lines

# Quick test on one train file
sample_xml = os.path.join(TRAIN_XML_DIR, sorted(os.listdir(TRAIN_XML_DIR))[0])
print("Testing on:", sample_xml)
print(voc_to_yolo_lines(sample_xml, CLASS_NAMES))


Testing on: C:\Users\HP\.cache\kagglehub\datasets\kanchisoni\drone-detection-dataset\versions\1\train\train\xml\00001.xml
['0 0.528182 0.485437 0.227273 0.218447']


## Cell 4 — Build the full YOLO dataset from train/ and test/ folders

In [4]:

import shutil

def build_split(img_dir, xml_dir, img_out_dir, lbl_out_dir, split_name):
    image_files = sorted([f for f in os.listdir(img_dir)
                           if f.lower().endswith((".jpg", ".jpeg", ".png"))])

    kept, skipped = 0, 0
    for fname in image_files:
        stem = os.path.splitext(fname)[0]
        xml_path = os.path.join(xml_dir, stem + ".xml")

        if not os.path.exists(xml_path):
            skipped += 1
            continue

        # copy image
        src_img = os.path.join(img_dir, fname)
        dst_img = os.path.join(img_out_dir, fname)
        shutil.copy2(src_img, dst_img)

        # write YOLO label
        lines = voc_to_yolo_lines(xml_path, CLASS_NAMES)
        label_path = os.path.join(lbl_out_dir, stem + ".txt")
        with open(label_path, "w") as f:
            f.write("\n".join(lines))

        kept += 1

    print(f"{split_name}: {kept} images processed, {skipped} skipped (no matching XML)")
    return kept

print("Building train split...")
n_train = build_split(TRAIN_IMG_DIR, TRAIN_XML_DIR, YOLO_IMAGES_TRAIN, YOLO_LABELS_TRAIN, "train")

print("\nBuilding val split...")
n_val = build_split(VAL_IMG_DIR, VAL_XML_DIR, YOLO_IMAGES_VAL, YOLO_LABELS_VAL, "val")

print(f"\nTotal: {n_train} train images, {n_val} val images")


Building train split...
train: 5200 images processed, 0 skipped (no matching XML)

Building val split...
val: 2600 images processed, 0 skipped (no matching XML)

Total: 5200 train images, 2600 val images


## Cell 5 — Create data.yaml

In [5]:

data_yaml_path = os.path.join(YOLO_DATASET_DIR, "data.yaml")

yaml_content = (
    f"path: {YOLO_DATASET_DIR}\n"
    "train: images/train\n"
    "val: images/val\n\n"
    "names:\n"
    "  0: UAV\n"
)

with open(data_yaml_path, "w") as f:
    f.write(yaml_content)

print("Saved:", data_yaml_path)
print(yaml_content)


Saved: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\data.yaml
path: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset
train: images/train
val: images/val

names:
  0: UAV



In [ ]:
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0%2Bcu124-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached torchaudio-2.6.0%2Bcu124-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
Using cached torch-2.6.0%2Bcu124-cp313-cp313-win_amd64.whl (2532.3 MB)
Using cached torchvision-0.21.0%2Bcu124-cp313-cp313-win_amd64.whl (6.1 MB)
Using cached torchaudio-2.6.0%2Bcu124-cp313-cp313-win_amd64.whl (4.2 MB)

   ---------------------------------------- 0/3 [torch]
   ------

## Cell 6 — Train YOLOv11 on GPU

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

CUDA available: True
Device count: 1


In [8]:

from ultralytics import YOLO

# yolo11s.pt = small pretrained checkpoint, fine-tuned on our UAV data
model = YOLO("yolo11s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=960,
    batch=8,          # can increase this if your GPU has enough VRAM (e.g. 16GB+)
    device=0,          # use first NVIDIA GPU
    patience=10,
    project=os.path.join(BASE_DIR, "yolo_runs"),
    name="uav_detect_full",
    exist_ok=True,
    seed=42,
)


Ultralytics 8.4.92  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ua

## Cell 7 — Evaluate on the val (test) set

In [9]:

best_weights = os.path.join(BASE_DIR, "yolo_runs", "uav_detect_full", "weights", "best.pt")
print("Using weights:", best_weights)

trained_model = YOLO(best_weights)
metrics = trained_model.val(data=data_yaml_path, imgsz=960, device=0)

print("\n===== YOLOv11 Full-Dataset Evaluation Summary =====")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print(f"mAP50:       {metrics.box.map50:.4f}")
print(f"mAP50-95:    {metrics.box.map:.4f}")


Using weights: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_runs\uav_detect_full\weights\best.pt
Ultralytics 8.4.92  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 489.6418.5 MB/s, size: 85.0 KB)
val: Scanning C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\labels\val.cache... 2600 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2600/2600 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 163/163 4.7it/s 34.6s0.2ss
                   all       2600       2621      0.963      0.903      0.945      0.638
Speed: 2.2ms preprocess, 7.9ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\HP\OneDrive\Documents\Desktop\vlm\scripts\runs\detect\val-2

===== YOLOv11 Full-D

## Cell 8 — Visualize predictions on a few val images

In [10]:

import matplotlib.pyplot as plt

val_images = sorted(os.listdir(YOLO_IMAGES_VAL))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, fname in enumerate(val_images):
    img_path = os.path.join(YOLO_IMAGES_VAL, fname)
    pred = trained_model.predict(img_path, imgsz=960, conf=0.25, verbose=False)[0]
    plotted = pred.plot()

    axes[i].imshow(plotted[..., ::-1])
    axes[i].set_title(fname)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


<Figure size 1500x1000 with 6 Axes>

## Cell 9 — Save summary report

In [11]:

from datetime import datetime

report_path = os.path.join(BASE_DIR, "yolo11_full_dataset_eval_summary.txt")

with open(report_path, "w") as f:
    f.write("YOLOv11 (fine-tuned, FULL Kaggle DUT Anti-UAV dataset) - Evaluation Report\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n")
    f.write(f"Base weights: yolo11s.pt (fine-tuned)\n")
    f.write(f"Train images: {n_train} | Val (test) images: {n_val}\n")
    f.write(f"Epochs: 30 | Image size: 960 | Device: GPU (cuda:0)\n\n")
    f.write(f"Precision:   {metrics.box.mp:.4f}\n")
    f.write(f"Recall:      {metrics.box.mr:.4f}\n")
    f.write(f"mAP50:       {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95:    {metrics.box.map:.4f}\n\n")
    f.write("NOTE: This run uses the full Kaggle DUT Anti-UAV Detection dataset\n")
    f.write("(thousands of images) instead of the original 50-image subset,\n")
    f.write("providing a more robust evaluation of YOLOv11 fine-tuning performance\n")
    f.write("compared to the small-sample run and the earlier VLM benchmarks.\n")

print("Summary report saved to:", report_path)


Summary report saved to: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo11_full_dataset_eval_summary.txt


## Cell 10 — Final held-out evaluation on test/ (never seen during training or validation)

In [12]:

TEST_YOLO_IMAGES = os.path.join(YOLO_DATASET_DIR, "images", "test")
TEST_YOLO_LABELS = os.path.join(YOLO_DATASET_DIR, "labels", "test")
os.makedirs(TEST_YOLO_IMAGES, exist_ok=True)
os.makedirs(TEST_YOLO_LABELS, exist_ok=True)

print("Building test split...")
n_test = build_split(TEST_IMG_DIR, TEST_XML_DIR, TEST_YOLO_IMAGES, TEST_YOLO_LABELS, "test")

# Add test path to data.yaml if not already present
with open(data_yaml_path, "r") as f:
    yaml_text = f.read()
if "test:" not in yaml_text:
    with open(data_yaml_path, "a") as f:
        f.write("test: images/test\n")

test_metrics = trained_model.val(data=data_yaml_path, imgsz=960, device=0, split="test")

print("\n===== Final Held-Out TEST Set Evaluation =====")
print(f"Precision:   {test_metrics.box.mp:.4f}")
print(f"Recall:      {test_metrics.box.mr:.4f}")
print(f"mAP50:       {test_metrics.box.map50:.4f}")
print(f"mAP50-95:    {test_metrics.box.map:.4f}")

# Append to summary report
with open(report_path, "a") as f:
    f.write("\n\n===== Final Held-Out TEST Set (unbiased, never used in training/val) =====\n")
    f.write(f"Test images: {n_test}\n")
    f.write(f"Precision:   {test_metrics.box.mp:.4f}\n")
    f.write(f"Recall:      {test_metrics.box.mr:.4f}\n")
    f.write(f"mAP50:       {test_metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95:    {test_metrics.box.map:.4f}\n")

print("\nTest results appended to:", report_path)


Building test split...
test: 2200 images processed, 0 skipped (no matching XML)
Ultralytics 8.4.92  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 29.417.7 MB/s, size: 139.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\labels\test... 2200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2200/2200 333.9it/s 6.6s0.1s
val: New cache created: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 138/138 4.7it/s 29.2s0.2s
                   all       2200       2245      0.959      0.939      0.969      0.692
Speed: 2.0ms preprocess, 8.1ms inferen